# Inductive Bias vs Context Effect Recovery — FETA on CDM, FATE on LCL

## The core question

The previous notebook left us with an uncomfortable result: self-attention recovers only ~60–70% of a planted context effect even when its predictive accuracy is essentially perfect. Prediction is fine; **identification** is not. The model gets the answer right without learning the mechanism.

There are two competing explanations for that, and they lead to very different fixes:

1. **Capacity story** — attention doesn't have enough room to represent the effect. Fix: bigger model, more layers.
2. **Inductive-bias story** — attention has plenty of room, but the cross-entropy objective gives it no reason to prefer the *true* functional form over any of the millions of other functions that fit the data equally well. Fix: change the architecture so the true form is the natural thing to learn.

This notebook tests story 2 directly.

> **Hypothesis.** A model whose architecture *matches* the true effect structure should recover that effect far better than a general-purpose attention model — not because it has more capacity, but because its functional form narrows the solution space in the right direction.

## The 2×2 design

The clean way to test this is to pair each synthetic data regime with the architecture built for that regime:

| Data | True effect structure | Matched architecture | General architecture |
|---|---|---|---|
| **CDM** | Pairwise: $\sum_{j \neq i} x_j^\top C\, x_i$ | **FETA** — score item *pairs*, then aggregate | Self-attention |
| **LCL** | Set-average: $\theta_{\text{eff}} = \theta + A\bar{x}_S$ | **FATE** — aggregate the *set* first, then score | Self-attention |

FETA is "First Evaluate Then Aggregate" and FATE is "First Aggregate Then Evaluate" (Pfannschmidt et al.). The names describe the order of operations, and that order is exactly what encodes the inductive bias. CDM's effect is fundamentally pairwise, so FETA's pair-then-sum order fits it. LCL's effect is fundamentally a set summary, so FATE's summarise-then-score order fits it.

Note that this is deliberately the *matched* half of the picture. The mismatched half (FETA on LCL, FATE on CDM) is a separate experiment; here we're asking whether matching helps at all before asking whether the match has to be exact.

## What gets measured

Same three metrics as the previous notebook, so results are directly comparable:

- **IIA ratio-shift recovery** — the headline. Scale-free, model-agnostic: regress the model's realised shift in $\log \frac{P(A)}{P(B)}$ against the closed-form true shift. Slope 1.0 = perfect recovery, 0.0 = the model behaves as if there were no context effect at all.
- **NLL gap vs yardstick** — test negative log-likelihood minus the correctly-specified model's test NLL. Near zero means "predicts as well as a model that knows the truth."
- **Old swap probe** — kept on the CDM side only, as a continuity line against the earlier notebook.

The pairing of the first two metrics is the whole point of the diagnostic. A model can sit at NLL gap ≈ 0 and IIA ≈ 0.6 simultaneously — that combination *is* the identification failure, stated in numbers.

## How to run this

Cells are strictly ordered and stateful: run top to bottom. Part 1 (CDM/FETA) is cells 1–10, Part 2 (LCL/FATE) is cells 11–19, and cell 20 merges both into one table. GPU is strongly recommended — on a Colab T4 expect roughly 5–8 minutes for the CDM sweeps and a similar amount for the LCL sweeps. The probes, not the training, are the slow part: each one runs thousands of single-slate forward passes.


---
## Part 1 — CDM data: FETA vs Self-Attention vs Yardstick

### Cell 1 — Imports and softmax

Standard setup, plus two small things worth naming.

**`softmax_np`** — subtracting the max before exponentiating is the numerical-stability trick. Mathematically $\text{softmax}(u) = \text{softmax}(u - c)$ for any constant $c$, because the constant cancels between numerator and denominator. Computationally it matters: with a context term that aggregates $d^2$ products across several neighbours, raw utilities can get large enough that `np.exp` overflows to `inf` and the probabilities come back as `nan`. Subtracting the max caps the largest exponent at $e^0 = 1$.

**`device`** — everything downstream calls `.to(device)`, so this one line decides whether the notebook runs in minutes or in an hour. If it prints `cpu` on Colab, go to *Runtime → Change runtime type → GPU* before continuing.

`mean` and `stdev` from `statistics` are imported here because the sweep functions aggregate across seeds with them rather than with numpy — no deep reason, just fewer array conversions.


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from statistics import mean, stdev

def softmax_np(u):
    u = u - np.max(u); e = np.exp(u); return e / e.sum()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Cell 2 — CDM data generator

This is the ground truth. Everything the notebook measures is measured *against* the matrix planted here, so it's worth being precise about what it plants.

The model of choice is:

$$U(i, S) = \underbrace{\theta^\top x_i}_{\text{own utility}} + \underbrace{\sum_{j \neq i} x_j^\top C\, x_i}_{\text{context: what the others do to } i}$$

then $P(i \mid S) = \text{softmax}(U)$ over the slate.

**Line by line:**

- **`theta`** — defaults to $[-1, 1, 0.5]$ tiled out to $d$ dimensions. One quirk to be aware of: when $d > 3$ the tiled vector is scaled by $0.8$, but when $d \le 3$ it isn't. So in the feature sweep (2 / 4 / 6) the base-utility weights are slightly smaller at $d = 4, 6$ than they would be by pure tiling. Minor, but it means the features panel isn't a perfectly clean "same problem, more dimensions" comparison.
- **`C`** — drawn from a *fixed* generator, `default_rng(12345)`, not from `seed`. That's deliberate and important: the ground truth stays identical across the three seeds, so seed-to-seed variation reflects sampling noise and training noise, not a moving target. Note that `C` for $d=2$ is not the top-left block of `C` for $d=4$ — each feature count gets its own random matrix.
- **`C_eff = context_strength * C`** — and `C_eff`, not `C`, is what goes into `gt`. The probes compare against the effective matrix, so setting `context_strength=0` gives a genuine no-context baseline where perfect recovery would be undefined rather than merely zero.
- **`M = X @ C_eff @ X.T`** — so `M[j, i]` $= x_j^\top C\, x_i$: the push item $j$ exerts on item $i$. Note this is *not* symmetric unless `C` happens to be.
- **`ctx = M.sum(axis=0) - np.diag(M)`** — summing down the columns gives $\sum_j x_j^\top C\, x_i$ for each $i$; subtracting the diagonal removes the $j = i$ self-term. This is the $\sum_{j \neq i}$ in the formula, done in two lines.
- **`ch = rng.choice(slate_size, p=P)`** — the choice is *sampled*, not taken as the argmax. This is what makes the problem statistical: even a model that knows $\theta$ and $C$ exactly cannot predict every click, and its test NLL is the irreducible floor that the yardstick will measure.

The output is long-format — one row per (session, item) with a `chosen` flag — which is the shape real click logs come in.

**Scale note for later.** The base term aggregates $d$ products; the context term aggregates roughly $4d^2$ of them. So as $d$ grows, the context term grows faster than the base term in absolute size while also becoming harder to pin down. Keep this in mind when reading the features panel: rising $d$ changes the signal mix, not just the parameter count.


In [ ]:
def generate_cdm_data(n_sessions=8000, n_features=3, slate_size=5,
                      theta=None, C=None, context_strength=1.0, seed=0):
    rng = np.random.default_rng(seed)
    d = n_features
    if theta is None:
        base = np.array([-1.0, 1.0, 0.5])
        theta = (np.tile(base, int(np.ceil(d/3)))[:d] * 0.8) if d > 3 else base[:d]
    theta = np.asarray(theta, float)
    if C is None:
        C = np.random.default_rng(12345).normal(0, 1, size=(d, d))
    C_eff = context_strength * np.asarray(C, float)

    rows = []
    for s in range(n_sessions):
        X   = rng.normal(0, 1, size=(slate_size, d))
        U   = X @ theta
        M   = X @ C_eff @ X.T
        ctx = M.sum(axis=0) - np.diag(M)
        P   = softmax_np(U + ctx)
        ch  = rng.choice(slate_size, p=P)
        for i in range(slate_size):
            row = {"session_id": s, "item_in_slate": i}
            for f in range(d): row[f"feat_{f}"] = X[i, f]
            row["chosen"] = int(i == ch)
            rows.append(row)
    gt = {"theta": theta, "C": C_eff, "n_features": d, "slate_size": slate_size}
    return pd.DataFrame(rows), gt

### Cell 3 — Collate and split (CDM)

Plumbing, but the tensor shapes here are the contract every model in Part 1 has to honour.

**`collate_slates`** groups the long dataframe by `session_id` and produces three aligned tensors:

| Tensor | Shape | Meaning |
|---|---|---|
| `X` | `(n_slates, max_slate, n_features)` | item feature vectors |
| `y` | `(n_slates, max_slate)` | one-hot: 1 for the chosen item |
| `mask` | `(n_slates, max_slate)` bool | `True` = real item, `False` = padding |

Every slate in this synthetic data has exactly 5 items, so no padding actually occurs and the mask is all-`True`. The machinery is kept anyway for two reasons: the models are written to be reusable on real data (Expedia slates vary in length), and the masking logic is where a subtle bug would hide if it were ever removed and re-added later. Cheap insurance.

**`split`** does a plain 80/20 split on a seeded permutation. Two details: the seed is passed through from the sweep so that each seed gets a different split as well as different data, and the split is at the *slate* level, not the row level — splitting rows would leak items from the same slate across train and test and quietly inflate test performance.


In [ ]:
def collate_slates(df, n_features):
    feat_cols = [f"feat_{f}" for f in range(n_features)]
    groups    = list(df.groupby("session_id"))
    n_slates  = len(groups)
    max_slate = df.groupby("session_id").size().max()
    X    = torch.zeros(n_slates, max_slate, n_features)
    y    = torch.zeros(n_slates, max_slate)
    mask = torch.zeros(n_slates, max_slate, dtype=torch.bool)
    for s, (_, g) in enumerate(groups):
        g = g.sort_values("item_in_slate"); k = len(g)
        X[s,:k]    = torch.tensor(g[feat_cols].values, dtype=torch.float32)
        y[s,:k]    = torch.tensor(g["chosen"].values,  dtype=torch.float32)
        mask[s,:k] = True
    return X, y, mask

def split(X, y, mask, frac=0.8, seed=0):
    n = X.shape[0]; g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g); n_tr = int(frac*n)
    tr, te = perm[:n_tr], perm[n_tr:]
    return (X[tr],y[tr],mask[tr]), (X[te],y[te],mask[te])

### Cell 4 — The three CDM models

This is the heart of Part 1. Three models see identical data and identical objectives; the only thing that varies is the shape of the function they're allowed to express.

**`AttentionChoiceModel`** — the general-purpose comparison. Embed each item with a linear layer, run a `TransformerEncoder` so every item attends to every other item, then score each contextualised representation with a linear head. Padded positions are set to $-\infty$ so softmax assigns them zero probability. There is deliberately **no positional encoding**: a slate is a set, and the model should give the same answer regardless of item order. Capacity here is far larger than the true effect needs — that's the point. If attention still under-recovers, capacity was never the constraint.

**`FETAChoiceModel`** — First Evaluate Then Aggregate. The forward pass builds every ordered pair in the slate and scores it:

- `xi = X.unsqueeze(2).expand(B,S,S,F)` → `xi[b,i,j]` is $x_i$ (repeated across $j$)
- `xj = X.unsqueeze(1).expand(B,S,S,F)` → `xj[b,i,j]` is $x_j$ (repeated across $i$)
- `torch.cat([xi, xj], dim=-1)` → shape `(B,S,S,2F)`, one concatenated pair per cell
- `pair_mlp(...)` → a scalar for every $(i,j)$ pair
- mask out the diagonal and any padded $j$, then `scores.sum(dim=2)` → sum over neighbours

Compare that last step to $\sum_{j \neq i} x_j^\top C\, x_i$ and the correspondence is exact **at the level of aggregation structure**: a per-pair quantity, summed over neighbours, added to an own-utility term (`own_score`, standing in for $\theta^\top x_i$).

Worth stating precisely, because the distinction matters for how we interpret the result: FETA's inductive bias fixes the *order of operations*, not the *functional form of the interaction*. The MLP still has to learn that the right pairwise function is bilinear. So FETA is not handed the answer — it's handed the right shape of question. It contains the true CDM effect in its function class (a two-hidden-layer MLP can approximate the bilinear form), plus a lot else besides.

Cost note: the pair MLP runs $S^2$ times per slate, so FETA is the slower of the two learned models even though it has fewer parameters.

**`CDMYardstick`** — the ceiling. Learns $\theta$ and $C$ directly, using the generator's formula verbatim. `torch.einsum("bjf,bif->bji", XC, X)` reproduces `M[b,j,i]` $= x_j^\top C\, x_i$, and the `sum over real j` minus `diagonal` reproduces the $j \neq i$ exclusion. Because it is correctly specified, its test NLL is the best any model can do given that choices were sampled rather than argmaxed. Both parameters are zero-initialised, which is fine here: the loss is well-conditioned in $(\theta, C)$ and the model has only $d + d^2$ parameters.


In [ ]:
class AttentionChoiceModel(nn.Module):
    def __init__(self, n_features=3, d_model=32, n_heads=4, n_layers=2, ff=64):
        super().__init__()
        self.embed   = nn.Linear(n_features, d_model)
        layer        = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                           dim_feedforward=ff, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.score   = nn.Linear(d_model, 1)

    def forward(self, X, mask):
        h      = self.embed(X)
        h      = self.encoder(h, src_key_padding_mask=~mask)
        logits = self.score(h).squeeze(-1)
        return logits.masked_fill(~mask, float("-inf"))


class FETAChoiceModel(nn.Module):
    """First-Evaluate-Then-Aggregate.
    For each item i, scores every (i,j) pair with a small MLP, then sums over j!=i.
    This is the pairwise inductive bias — structurally the same order as CDM's C matrix.
    Architecture: MLP(concat(x_i, x_j)) -> scalar, summed over j.
    """
    def __init__(self, n_features=3, hidden=32):
        super().__init__()
        # pairwise MLP: takes concat of two items (2*n_features) -> scalar score
        self.pair_mlp = nn.Sequential(
            nn.Linear(2 * n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        # own-utility: score for item i on its own (the theta.x_i part)
        self.own_score = nn.Linear(n_features, 1)

    def forward(self, X, mask):
        B, S, F = X.shape
        # own-utility term: (B, S)
        own = self.own_score(X).squeeze(-1)

        # pairwise term: for each (i,j) pair, score MLP(concat(x_i, x_j))
        # expand to get all pairs: xi (B,S,1,F), xj (B,1,S,F)
        xi = X.unsqueeze(2).expand(B, S, S, F)   # (B, S, S, F) — repeated along dim2
        xj = X.unsqueeze(1).expand(B, S, S, F)   # (B, S, S, F) — repeated along dim1
        pairs = torch.cat([xi, xj], dim=-1)       # (B, S, S, 2F)
        scores = self.pair_mlp(pairs).squeeze(-1)  # (B, S, S)

        # mask out padding and the diagonal (j != i)
        diag = torch.eye(S, dtype=torch.bool, device=X.device).unsqueeze(0)
        pad_j = ~mask.unsqueeze(1).expand(B, S, S)  # (B, S, S)
        scores = scores.masked_fill(diag | pad_j, 0.0)

        # sum over j for each i -> pairwise context term (B, S)
        context = scores.sum(dim=2)

        logits = (own + context).masked_fill(~mask, float("-inf"))
        return logits


class CDMYardstick(nn.Module):
    """Correctly-specified CDM ceiling. Learns theta and C directly."""
    def __init__(self, n_features=3):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(n_features))
        self.C     = nn.Parameter(torch.zeros(n_features, n_features))

    def forward(self, X, mask):
        base  = X @ self.theta
        XC    = X @ self.C
        M     = torch.einsum("bjf,bif->bji", XC, X)
        realj = mask.unsqueeze(-1).float()
        ctx   = (M * realj).sum(dim=1) - torch.diagonal(M, dim1=1, dim2=2)
        return (base + ctx).masked_fill(~mask, float("-inf"))

### Cell 5 — Training loop (returns test NLL)

Same loop as the previous notebook, unchanged, so numbers stay comparable.

The mechanics: all tensors are moved to the device once up front (the datasets are small enough to sit in GPU memory whole, which avoids per-batch transfer overhead), targets are extracted as `y.argmax(1)` — the index of the chosen item — and training is plain minibatch Adam on `cross_entropy`.

Two things worth noticing.

**Cross-entropy over a slate *is* the choice likelihood.** `F.cross_entropy(logits, target)` applies a softmax over the slate dimension and takes the negative log-probability of the chosen index. That's exactly $-\log P(i^* \mid S)$ under the model. This is not "classification as an approximation to choice modelling" — it is the same objective written in PyTorch's vocabulary. The $-\infty$ logits at padded positions drop out of the softmax cleanly.

**The returned NLL is the last epoch's, not the best epoch's.** No early stopping, no checkpoint selection. For these models and epoch counts that's fine — the curves are flat by the end — but it's a modelling choice, not an oversight, and it's worth knowing if a config ever comes back with a surprising NLL gap.

Accuracy is computed for the `verbose` printout only. It's a reassurance metric rather than a diagnostic one: high accuracy alongside poor IIA recovery is precisely the pattern this notebook exists to demonstrate.


In [ ]:
def train_model(model, Xtr, ytr, mtr, Xte, yte, mte,
                epochs=30, lr=1e-3, batch=256, seed=0, verbose=False):
    model = model.to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr,ytr,mtr = Xtr.to(device),ytr.to(device),mtr.to(device)
    Xte,yte,mte = Xte.to(device),yte.to(device),mte.to(device)
    tgt_tr = ytr.argmax(1); tgt_te = yte.argmax(1)
    n = Xtr.shape[0]; g = torch.Generator().manual_seed(seed)
    te_loss = float("nan")
    for ep in range(1, epochs+1):
        model.train()
        perm = torch.randperm(n, generator=g)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            loss = F.cross_entropy(model(Xtr[idx], mtr[idx]), tgt_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            te_loss = F.cross_entropy(model(Xte,mte), tgt_te).item()
            acc     = (model(Xte,mte).argmax(1)==tgt_te).float().mean().item()
        if verbose and (ep%5==0 or ep==1):
            print(f"ep {ep:2d} | test_nll {te_loss:.4f} | acc {acc:.3f}")
    return model, te_loss

### Cell 6 — IIA ratio-shift and old swap-probe (CDM versions)

Copied unchanged from the previous CDM notebook. Both probes only need a callable taking `(X, mask)` and returning logits, so they work on attention, FETA, and the yardstick identically — which is exactly what makes them a fair basis for comparison.

#### `iia_ratio_recovery_cdm` — the headline metric

Independence of Irrelevant Alternatives says the ratio $P(A)/P(B)$ shouldn't change when some third item $C$ changes. Context effects break that, and the size of the break is what we measure.

The construction:

1. Draw two focal items $x_A, x_B$ and some fillers. Hold them fixed.
2. Build two slates that differ **only in the last item**: one with $x_{C_1}$, one with $x_{C_2}$.
3. `ab_gap` returns $\text{logit}_A - \text{logit}_B$, which is $\log \frac{P(A)}{P(B)}$ (the softmax normaliser cancels in the difference).
4. `d_mod` is how much that log-ratio moved between the two slates.

The truth is available in closed form. Under CDM, the $A$–$B$ gap picks up $x_C^\top C\, (x_A - x_B)$ from the swapped item, and every other term — $\theta$ contributions, filler pushes, the $A \leftrightarrow B$ mutual push — is identical across the two slates and cancels. So:

$$d_{\text{true}} = (x_{C_2} - x_{C_1})^\top C\, (x_A - x_B)$$

Regressing `d_mod` on `d_true` across 3000 trials and taking the slope gives the recovery coefficient: **1.0 = perfect**, **0.0 = the model behaves as if IIA held**, negative = the model moves the ratio the wrong way. The correlation is returned alongside and answers a different question — slope is "how much of the effect", correlation is "is it even tracking the right direction". A high correlation with a low slope means systematic shrinkage; a low correlation means the model isn't tracking the effect at all.

This metric is scale-free and requires no assumption about the model's internals, which is why it's the one to lead with in the write-up.

#### `old_swap_slope` — the continuity line

The earlier, cruder probe: hold $x_i$ fixed, swap one neighbour from $x_{jA}$ to $x_{jB}$, and see how item $i$'s raw logit moves. It's kept purely so results line up against the earlier notebook.

Two honest notes on this function. First, the `d_true_arr` block with the walrus operator is **dead code** — it's computed and then immediately superseded by the `dt2` loop below it. It doesn't corrupt anything (its generators are local), it just wastes time. Second, `dt2` reconstructs the true shifts by re-seeding a generator and redrawing in *exactly* the same order as the first loop (`xi`, then fillers, then `xjA`, then `xjB`). That works, but it's fragile: reorder a single draw in the first loop and the two halves silently desynchronise. This is a good argument for treating the IIA probe as the primary metric and this one as a legacy cross-check.


In [ ]:
def iia_ratio_recovery_cdm(model, C_true, n_features, slate_size=5,
                           n_trials=3000, seed=7):
    rng = np.random.default_rng(seed); model.eval()
    C_true = np.asarray(C_true)
    d_true_all, d_mod_all = [], []
    def ab_gap(slate):
        X = torch.tensor(np.vstack(slate), dtype=torch.float32, device=device).unsqueeze(0)
        m = torch.ones(1, len(slate), dtype=torch.bool, device=device)
        with torch.no_grad(): lg = model(X, m)[0].cpu().numpy()
        return lg[0] - lg[1]
    for _ in range(n_trials):
        x_A = rng.normal(0,1,size=n_features); x_B = rng.normal(0,1,size=n_features)
        fillers = [rng.normal(0,1,size=n_features) for _ in range(slate_size-3)]
        x_C1 = rng.normal(0,1,size=n_features); x_C2 = rng.normal(0,1,size=n_features)
        base = [x_A, x_B] + fillers
        d_mod  = ab_gap(base+[x_C2]) - ab_gap(base+[x_C1])
        d_true = (x_C2-x_C1) @ (C_true @ (x_A-x_B))
        d_true_all.append(d_true); d_mod_all.append(d_mod)
    d_true_all = np.array(d_true_all); d_mod_all = np.array(d_mod_all)
    return np.polyfit(d_true_all, d_mod_all, 1)[0], np.corrcoef(d_true_all, d_mod_all)[0,1]

def old_swap_slope(model, C_true, n_features, slate_size=5, n_trials=3000, seed=1):
    rng = np.random.default_rng(seed); model.eval(); C_true = np.asarray(C_true)
    dt, dm = [], []
    with torch.no_grad():
        for _ in range(n_trials):
            xi = rng.normal(0,1,n_features); fill = rng.normal(0,1,(slate_size-2,n_features))
            xjA = rng.normal(0,1,n_features); xjB = rng.normal(0,1,n_features)
            d_true = (xjB-xjA) @ (C_true @ xi)
            for xj, lst in [(xjA, dt), (xjB, dm)]:
                sl = np.vstack([xi,xj,fill])
                X  = torch.tensor(sl,dtype=torch.float32,device=device).unsqueeze(0)
                mk = torch.ones(1,slate_size,dtype=torch.bool,device=device)
                lst.append(model(X,mk)[0,0].item())
    d_attn = np.array(dm)-np.array(dt)
    d_true_arr = np.array([((rng2:=np.random.default_rng(1+i)).normal(0,1,n_features)-
                             rng2.normal(0,1,n_features)) @ (C_true @ rng2.normal(0,1,n_features))
                            for i in range(n_trials)])
    # simpler: just recompute true shifts directly
    rng2 = np.random.default_rng(seed)
    dt2 = []
    for _ in range(n_trials):
        xi=rng2.normal(0,1,n_features); rng2.normal(0,1,(slate_size-2,n_features))
        xjA=rng2.normal(0,1,n_features); xjB=rng2.normal(0,1,n_features)
        dt2.append((xjB-xjA)@(C_true@xi))
    dt2=np.array(dt2)
    slope = np.polyfit(dt2, d_attn, 1)[0]
    return slope

### Cell 7 — Smoke test: all three CDM models on the default config

Before committing 5–8 minutes to sweeps, confirm the wiring on one config: 8000 sessions, 3 features, full context strength.

Note the deliberately different training budgets. Attention gets 30 epochs at `lr=1e-3`; FETA gets 40 (its pairwise MLP has more to learn per step and converges a little slower); the yardstick gets 100 epochs at `lr=5e-2` because it has only $d + d^2 = 12$ parameters and a large learning rate finds them fast. These are not tuned for fairness in a "same compute" sense — they're tuned so each model reaches its own convergence, which is the relevant notion of fairness for a recovery question.

**What to check in the output, in order of importance:**

1. **Does FETA's IIA beat attention's?** If the inductive-bias hypothesis is right, this should already be visible here, before any sweeping. A gap of a few percentage points is noise; a gap of 0.2+ is the effect.
2. **Is the yardstick's IIA near 1.0?** It should be. If it isn't, the probe and the generator disagree about the ground truth and every other number in the notebook is suspect. This is the assertion that validates the measurement instrument itself.
3. **Are the NLL gaps small?** Both learned models should predict nearly as well as the yardstick. Small NLL gap *plus* sub-1.0 IIA recovery is the identification failure in a single printout.

If the yardstick's NLL comes out *worse* than attention's, something is wrong with its optimisation (try more epochs) — a correctly-specified model should not be beaten on likelihood by a misspecified one at this sample size.


In [ ]:
df0, gt0 = generate_cdm_data(n_sessions=8000, n_features=3,
                              context_strength=1.0, seed=0)
(Xtr0,ytr0,mtr0),(Xte0,yte0,mte0) = split(*collate_slates(df0, 3))

results_smoke = {}
for name, model, epochs, lr in [
    ("Attention", AttentionChoiceModel(n_features=3, n_layers=2), 30, 1e-3),
    ("FETA",      FETAChoiceModel(n_features=3, hidden=32),       40, 1e-3),
    ("Yardstick", CDMYardstick(n_features=3),                    100, 5e-2),
]:
    torch.manual_seed(0)
    m, nll = train_model(model, Xtr0,ytr0,mtr0, Xte0,yte0,mte0,
                         epochs=epochs, lr=lr, verbose=False)
    iia, _ = iia_ratio_recovery_cdm(m, gt0["C"], 3)
    results_smoke[name] = {"nll": nll, "iia": iia}
    print(f"{name:12s}  test_nll={nll:.4f}  iia_recovery={iia:.3f}")

nll_yard = results_smoke["Yardstick"]["nll"]
print(f"\nNLL gap vs yardstick:")
for name in ["Attention","FETA"]:
    print(f"  {name}: {results_smoke[name]['nll']-nll_yard:+.4f}")

### Cell 8 — CDM sweep engine (three models)

Turns the smoke test into a systematic experiment.

**`run_cdm_config`** — one full experiment at one configuration: generate data, split it, train all three models, and probe the two learned ones. `torch.manual_seed(seed)` is reset before each model so initialisation is controlled, and the probe seeds are offset (`200+seed` for IIA, `100+seed` for the old probe) so the probe's random slates are independent of the training data draw. That independence matters — probing on slates correlated with the training set would measure memorisation rather than the learned mechanism.

The yardstick is trained at every configuration too, but only its NLL is used. That's the point of it: it defines the moving floor that the NLL gap is measured against, and that floor shifts with data size and feature count.

**`cdm_sweep`** — loops over the swept values, runs all three seeds at each, and accumulates mean and standard deviation for each metric. The running printout after each value is there so a long sweep can be monitored rather than watched in silence.

**The three axes and what each one asks:**

| Axis | Values | Question |
|---|---|---|
| `n_sessions` | 10k / 15k / 20k | Is under-recovery a data-starvation problem? If recovery climbs steadily with data, it's a sample-size issue, not an identification issue. |
| `n_features` | 2 / 4 / 6 | How does recovery degrade with dimensionality? The earlier finding was "sharply" for attention — does FETA's bias hold up better? |
| `n_layers` | 1 / 2 / 3 / 4 / 6 | Is it a capacity problem? More layers means strictly more capacity. Flat or declining recovery here kills the capacity story. |

The depth sweep has a useful property built in: **FETA has no layers**, so its `hidden=32` config is identical across all five points. Its line should therefore be flat. That flat line is a free control — if FETA's curve wanders on the depth axis, the wander is seed noise, and it tells you how much of any wander in attention's curve is also just noise.

Runtime is dominated by the probes, not by training: `old_swap_slope` alone does 6000 single-slate forward passes per model per seed. Dropping it would roughly halve the sweep time if you ever need the CDM sweeps to run faster.


In [ ]:
SEEDS = [0, 1, 2]

def run_cdm_config(n_sessions, n_features, n_layers, seed):
    df, gt = generate_cdm_data(n_sessions=n_sessions, n_features=n_features,
                               context_strength=1.0, seed=seed)
    (Xtr,ytr,mtr),(Xte,yte,mte) = split(*collate_slates(df, n_features), seed=seed)

    out = {}
    configs = [
        ("Attention", AttentionChoiceModel(n_features=n_features, n_layers=n_layers), 30, 1e-3),
        ("FETA",      FETAChoiceModel(n_features=n_features, hidden=32),              40, 1e-3),
        ("Yardstick", CDMYardstick(n_features=n_features),                           100, 5e-2),
    ]
    for name, model, epochs, lr in configs:
        torch.manual_seed(seed)
        m, nll = train_model(model, Xtr,ytr,mtr, Xte,yte,mte,
                             epochs=epochs, lr=lr, seed=seed)
        iia, _ = iia_ratio_recovery_cdm(m, gt["C"], n_features, seed=200+seed)
        slope   = old_swap_slope(m, gt["C"], n_features, seed=100+seed)
        out[name] = {"nll": nll, "iia": iia, "slope": slope}
    return out, gt["C"]

def cdm_sweep(values, vary, base):
    # returns dict: model_name -> {iia: ([means],[stds]), nll_gap: ..., slope: ...}
    accum = {m: {"iia":([],[]), "nll_gap":([],[]), "slope":([],[])}
             for m in ["Attention","FETA"]}
    xs = []
    for v in values:
        cfg = dict(base); cfg[vary] = v
        per = {m: {"iia":[],"nll_gap":[],"slope":[]} for m in ["Attention","FETA"]}
        for sd in SEEDS:
            res, _ = run_cdm_config(seed=sd, **cfg)
            nll_y  = res["Yardstick"]["nll"]
            for m in ["Attention","FETA"]:
                per[m]["iia"].append(res[m]["iia"])
                per[m]["nll_gap"].append(res[m]["nll"] - nll_y)
                per[m]["slope"].append(res[m]["slope"])
        xs.append(v)
        for m in ["Attention","FETA"]:
            for k in ["iia","nll_gap","slope"]:
                accum[m][k][0].append(mean(per[m][k]))
                accum[m][k][1].append(stdev(per[m][k]) if len(per[m][k])>1 else 0.)
        print(f"{vary}={v} | Attn iia={accum['Attention']['iia'][0][-1]:.3f} "
              f"| FETA iia={accum['FETA']['iia'][0][-1]:.3f}")
    return xs, accum

### Cell 9 — Run the CDM sweeps

Three sweeps, one per axis, all sharing the same base configuration so the panels are commensurable. Around 5–8 minutes on a Colab GPU.

The printed lines give a live read on the headline comparison — `Attn iia` versus `FETA iia` at each value — so the answer to the main question is visible before the plots render. Watch the depth sweep in particular: that's where the capacity story lives or dies.

If a sweep needs to be cut short for time, the features sweep is the one to keep. It carries the most information, because the earlier finding (sharp degradation with dimensionality) is the sharpest thing this notebook is trying to explain.


In [ ]:
print("=== CDM SWEEP 1: dataset size ===")
xs_s, res_s = cdm_sweep([10000,15000,20000], "n_sessions",
                        {"n_features":3,"n_layers":2})

print("\n=== CDM SWEEP 2: n_features ===")
xs_f, res_f = cdm_sweep([2,4,6], "n_features",
                        {"n_sessions":15000,"n_layers":2})

print("\n=== CDM SWEEP 3: n_layers (attention only, FETA stays fixed) ===")
xs_l, res_l = cdm_sweep([1,2,3,4,6], "n_layers",
                        {"n_sessions":15000,"n_features":3})

### Cell 10 — CDM comparison plots

Three panels, one per swept axis. Each panel carries four things:

- **Solid lines with circles, left axis** — IIA recovery. FETA in blue (`C0`), Attention in orange (`C1`). Error bars are ±1 standard deviation over the three seeds.
- **Dotted green horizontal at 1.0** — perfect recovery, where the yardstick sits by construction.
- **Dashed lines with triangles, right axis** — NLL gap vs yardstick, same colours, faded. Closer to zero is better predictively.

**What confirms the hypothesis:** the blue line sitting consistently above the orange one across all three panels. Ideally with the dashed lines nearly on top of each other — that combination says "both models predict about equally well, but only one of them learned the mechanism," which is the cleanest possible statement of the result.

**Two reading caveats.**

`ax.set_ylim(-0.05, 1.15)` clips the IIA axis. If a marker seems to be missing from a panel, it hasn't failed to compute — it's outside the window. Negative recovery is a real possible outcome (attention on LCL has produced it before), so widen the limits before concluding a point is absent.

The figure legend is built from `axes[0].get_legend_handles_labels()`, which only sees the *primary* axis. The dashed NLL-gap lines live on the twin axis and therefore **don't appear in the legend**. They're identifiable by linestyle and by the grey right-hand axis label, but it's worth knowing rather than discovering mid-presentation.

The PNG is saved at 150 dpi with tight bounding box, which is thesis-figure quality as-is.


In [ ]:
import matplotlib.pyplot as plt

COLORS = {"Attention": "C1", "FETA": "C0", "FATE": "C2"}

def plot_cdm_panel(ax, xs, accum, xlabel, title):
    for name, color in [("Attention","C1"),("FETA","C0")]:
        ax.errorbar(xs, accum[name]["iia"][0], yerr=accum[name]["iia"][1],
                    marker="o", label=f"{name} — IIA recovery", color=color)
    ax.axhline(1.0, ls=":", c="green", lw=1.2, label="perfect recovery (yardstick)")
    ax.set_xlabel(xlabel); ax.set_ylabel("IIA ratio recovery")
    ax.set_title(title); ax.set_ylim(-0.05, 1.15)
    ax2 = ax.twinx()
    for name, color in [("Attention","C1"),("FETA","C0")]:
        ax2.errorbar(xs, accum[name]["nll_gap"][0], yerr=accum[name]["nll_gap"][1],
                     marker="^", ls="--", color=color, alpha=0.5,
                     label=f"{name} — NLL gap")
    ax2.set_ylabel("NLL gap vs yardstick (nats)", color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")
    return ax2

fig, axes = plt.subplots(1,3,figsize=(18,5))
plot_cdm_panel(axes[0], xs_s, res_s, "n_sessions", "CDM: recovery vs data size")
plot_cdm_panel(axes[1], xs_f, res_f, "n_features",  "CDM: recovery vs #features")
plot_cdm_panel(axes[2], xs_l, res_l, "n_layers",    "CDM: recovery vs attn depth")

handles,labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5,-0.08))
plt.suptitle("CDM synthetic — FETA (matched inductive bias) vs Self-Attention", y=1.02)
plt.tight_layout()
plt.savefig("cdm_feta_vs_attn.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved cdm_feta_vs_attn.png")

---
## Part 2 — LCL data: FATE vs Self-Attention vs Yardstick

Part 1 asked whether a *pairwise* inductive bias helps on *pairwise* effects. Part 2 asks the same question for a completely different effect structure, and that's what makes the pair of results an argument rather than an anecdote. If matching helps in both regimes, the claim is about matching in general — not about FETA being a good model.

### Cell 11 — LCL data generator and theta/A builder

LCL (Linear Context Logit) plants a **first-order, set-average** effect. Instead of items pushing on each other pairwise, the whole slate produces a summary and that summary bends the preference weights:

$$\theta_{\text{eff}}(S) = \theta + A\,\bar{x}_S, \qquad U(i,S) = \theta_{\text{eff}}(S)^\top x_i$$

where $\bar{x}_S$ is the mean feature vector of the slate.

**`make_theta_A`** builds a deliberately minimal ground truth. $\theta$ is $[-0.5, 1.0]$ tiled to length $d$, and $A$ is **all zeros except one entry**: `A[d-1, d-1] = 0.3`. So the entire context effect is:

$$\Delta U_i = 0.3 \cdot x_{i,\text{last}} \cdot \overline{x}_{\text{last}}$$

In words: when the slate as a whole scores high on the last feature, the weight the chooser places on that feature increases. Think of a hotel slate where every option is expensive — price stops being a distinguishing feature and the chooser starts weighting it differently than they would in a mixed-price slate. A single scalar, one interaction, nothing else. If a model can't find *this*, it can't find anything.

**`generate_lcl_data`** then does per session: draw $X$, compute `xC = X.mean(axis=0)`, form `U = X @ (theta + A @ xC)`, softmax, sample.

One detail worth flagging: **`xC` includes item $i$ itself.** The mean is over the whole slate, not over the neighbours. That's a real modelling choice, and it means item $i$ contributes $1/S$ of its own context. The probe in cell 15 handles this correctly by computing the truth from realised slate means rather than from the nominal shift, so nothing downstream breaks — but it's the kind of thing that causes a mysterious slope of 0.8 instead of 1.0 if it ever gets mishandled.

Note also the column name here is `item_id_in_session`, where the CDM generator used `item_in_slate`. The two halves of the notebook come from different lineages and keep their own conventions; the two collate functions are the reason this doesn't matter.


In [ ]:
def softmax_lcl(u):
    u = u - np.max(u); e = np.exp(u); return e / e.sum()

def make_theta_A(n_features, a_strength=0.3):
    base  = np.array([-0.5, 1.0])
    theta = np.tile(base, int(np.ceil(n_features/2)))[:n_features].astype(float)
    A     = np.zeros((n_features, n_features))
    A[n_features-1, n_features-1] = a_strength
    return theta, A

def generate_lcl_data(n_sessions=5000, slate_size=5, n_features=2,
                      theta=None, A=None, seed=0):
    rng = np.random.default_rng(seed)
    if theta is None or A is None: theta, A = make_theta_A(n_features)
    theta = np.asarray(theta,float); A = np.asarray(A,float)
    rows = []
    for s in range(n_sessions):
        X   = rng.standard_normal((slate_size, n_features))
        xC  = X.mean(axis=0)
        U   = X @ (theta + A @ xC)
        P   = softmax_lcl(U)
        ch  = rng.choice(slate_size, p=P)
        for i in range(slate_size):
            row = {"session_id": s, "item_id_in_session": i}
            for f in range(n_features): row[f"feat_{f}"] = X[i,f]
            row["chosen"] = int(i==ch)
            rows.append(row)
    return pd.DataFrame(rows)

### Cell 12 — SlateDataset and collate_fn (LCL versions)

The LCL side uses PyTorch's `Dataset`/`DataLoader` interface rather than the whole-tensor approach from Part 1. Both are reproduced from their original notebooks so this file is self-contained and neither set of results has to be re-derived.

**`SlateDataset`** accepts either a path or a dataframe (handy for reloading a saved CSV without changing the call site), infers `feat_cols` from the `feat_` prefix — so `n_features` is discovered rather than declared, which is why the training loop takes a *builder* rather than a model — and stores each session as an `(X, chosen_index)` tuple.

**`collate_lcl`** pads a batch to the longest slate in that batch and returns `(features, mask, chosen)`.

The one thing to notice: the tuple order differs from Part 1. CDM's collate returns `(X, y, mask)` with a one-hot `y`; LCL's returns `(features, mask, chosen)` with an integer index. Same information, different packaging. This mismatch is the entire reason two training loops exist instead of one, and it's the most likely source of a confusing error if code is ever copied between the halves.


In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split

class SlateDataset(Dataset):
    def __init__(self, source):
        df = pd.read_csv(source) if isinstance(source,str) else source
        self.feat_cols  = [c for c in df.columns if c.startswith("feat_")]
        self.n_features = len(self.feat_cols)
        self.slates = []
        for _, g in df.groupby("session_id"):
            g = g.sort_values("item_id_in_session")
            X = g[self.feat_cols].to_numpy(dtype=np.float32)
            ch = int(np.argmax(g["chosen"].to_numpy()))
            self.slates.append((X, ch))
    def __len__(self):  return len(self.slates)
    def __getitem__(self,i): return self.slates[i]

def collate_lcl(batch):
    sizes = [X.shape[0] for X,_ in batch]; S = max(sizes); F = batch[0][0].shape[1]
    B = len(batch)
    features = torch.zeros(B,S,F,dtype=torch.float32)
    mask     = torch.zeros(B,S,dtype=torch.bool)
    chosen   = torch.zeros(B,dtype=torch.long)
    for b,(X,idx) in enumerate(batch):
        s = X.shape[0]
        features[b,:s] = torch.from_numpy(X)
        mask[b,:s]     = True
        chosen[b]      = idx
    return features, mask, chosen

### Cell 13 — The three LCL models

Same three-way structure as cell 4, with the matched architecture swapped for the one that fits LCL.

**`AttentionChoiceModelLCL`** — identical in design to Part 1's attention model, just re-declared with LCL-appropriate defaults (`n_features=2`, `n_layers=1`). Still permutation-equivariant, still far more capacity than the true effect needs.

**`FATEChoiceModel`** — First Aggregate Then Evaluate. Three steps in the forward pass:

1. **Masked set mean.** `x_C = (X * m_float).sum(1) / m_float.sum(1)` — sum only real items, divide only by the count of real items. Using a plain `.mean(1)` would drag padded zeros into the average and quietly bias every context summary on variable-length slates.
2. **Broadcast.** `x_C.unsqueeze(1).expand_as(X)` gives every item its own copy of the same set summary.
3. **Score.** `item_mlp(cat([X, x_C_exp]))` → one scalar per item.

The bias this encodes: *an item's score may depend on the item and on a summary of the set, and on nothing else about which specific items are in the set.* That is exactly the conditioning structure of $\theta + A\bar{x}_S$.

To be precise about what's being claimed — the same precision as with FETA — the match is at the level of **conditioning structure**, not functional form. FATE is handed $\bar{x}_S$ explicitly, in the right place, at the right time; but the MLP still has to learn that the correct way to combine $x_i$ with $\bar{x}_S$ is the *product* $x_{i,\text{last}} \cdot \bar{x}_{\text{last}}$. Concatenation makes that product learnable, not given. So FATE is not the yardstick in disguise. It's a model that has been pointed at the right question.

This is also precisely what attention has to discover on its own: attention *can* compute something like a set average (uniform attention weights over the slate would do it), but nothing in the architecture or the objective pushes it to.

**`LCLYardstick`** — correctly specified. `theta_eff = self.theta + x_C @ self.A.T` gives row $i$ the value $\theta + A\bar{x}_S$, and `(X * theta_eff.unsqueeze(1)).sum(-1)` is the per-item dot product. Same masked-mean computation as FATE, deliberately — if the mean were computed differently in the two models, part of any gap between them would be an artefact of that difference rather than of the architecture.


In [ ]:
class AttentionChoiceModelLCL(nn.Module):
    def __init__(self, n_features=2, hidden=32, n_heads=4, n_layers=1, ff=64):
        super().__init__()
        self.embed   = nn.Linear(n_features, hidden)
        layer        = nn.TransformerEncoderLayer(d_model=hidden, nhead=n_heads,
                           dim_feedforward=ff, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.score   = nn.Linear(hidden, 1)

    def forward(self, X, mask):
        h      = self.embed(X)
        h      = self.encoder(h, src_key_padding_mask=~mask)
        logits = self.score(h).squeeze(-1)
        return logits.masked_fill(~mask, float("-inf"))


class FATEChoiceModel(nn.Module):
    """First-Aggregate-Then-Evaluate.
    Computes the masked set mean x_C, then scores each item as MLP(concat(x_i, x_C)).
    This is the set-average inductive bias — structurally identical to LCL's theta + A x_C.
    """
    def __init__(self, n_features=2, hidden=32):
        super().__init__()
        # item MLP: takes concat(x_i, x_C) and outputs a scalar utility
        self.item_mlp = nn.Sequential(
            nn.Linear(2 * n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, X, mask):
        # masked set mean: x_C = mean of real items only
        m_float = mask.unsqueeze(-1).float()              # (B,S,1)
        x_C = (X * m_float).sum(dim=1) / m_float.sum(dim=1)  # (B, F)

        # expand x_C to match each item: (B, S, F)
        x_C_exp = x_C.unsqueeze(1).expand_as(X)

        # score each item given the set context
        concat = torch.cat([X, x_C_exp], dim=-1)          # (B, S, 2F)
        logits = self.item_mlp(concat).squeeze(-1)         # (B, S)
        return logits.masked_fill(~mask, float("-inf"))


class LCLYardstick(nn.Module):
    """Correctly-specified LCL ceiling. Learns theta and A directly."""
    def __init__(self, n_features=2):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(n_features))
        self.A     = nn.Parameter(torch.zeros(n_features, n_features))

    def forward(self, X, mask):
        m_float  = mask.unsqueeze(-1).float()
        x_C      = (X * m_float).sum(dim=1) / m_float.sum(dim=1)
        theta_eff = self.theta + x_C @ self.A.T
        logits   = (X * theta_eff.unsqueeze(1)).sum(-1)
        return logits.masked_fill(~mask, float("-inf"))

### Cell 14 — LCL training loop (DataLoader version)

Functionally the same job as cell 5's loop, restructured around the `Dataset` interface.

**Why a builder, not a model.** `train_lcl_model` takes `builder`, a function of `n_features`, and calls it *after* the dataset is constructed. That's because `SlateDataset` infers the feature count from the dataframe columns, so the model can't be instantiated until the data exists. It also guarantees a fresh model per call — passing a pre-built model into a sweep loop is a classic way to accidentally continue training an already-trained network across configurations.

**Mechanics.** `random_split` does the 80/20 with its own seeded generator, `torch.manual_seed(seed)` at the top controls initialisation, and the inner `run` closure handles both train and eval passes — hence the `if train:` guards around `zero_grad`, `backward`, and `step`.

**One detail that's easy to get wrong.** The loss accumulation is `tot += loss.item() * len(chosen)` and returns `tot / n`. `F.cross_entropy` averages over the batch by default, so multiplying back by the batch size and dividing by the total recovers a true per-example mean. A plain average of batch losses would over-weight the final, possibly-smaller batch. Since the NLL gap is a comparison between two numbers of this kind, a systematic bias here would propagate straight into the headline metric.

As in Part 1, the returned NLL is the final epoch's, with no early stopping.


In [ ]:
def train_lcl_model(builder, ds, epochs=30, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    n_test  = int(0.2*len(ds)); n_train = len(ds)-n_test
    tr_ds, te_ds = random_split(ds,[n_train,n_test],
                                generator=torch.Generator().manual_seed(seed))
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_lcl)
    te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_lcl)
    model = builder(ds.n_features).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    def run(loader, train):
        model.train() if train else model.eval()
        tot,n = 0.,0
        for feat,mask,chosen in loader:
            feat,mask,chosen = feat.to(device),mask.to(device),chosen.to(device)
            if train: opt.zero_grad()
            loss = F.cross_entropy(model(feat,mask), chosen)
            if train: loss.backward(); opt.step()
            tot += loss.item()*len(chosen); n += len(chosen)
        return tot/n

    te = float("nan")
    for ep in range(1, epochs+1):
        run(tr_loader, True)
        with torch.no_grad(): te = run(te_loader, False)
    return model, te

### Cell 15 — IIA ratio-shift probe (LCL version)

Same principle as the CDM probe — measure how much the model shifts $\log \frac{P(A)}{P(B)}$ and compare against the closed-form truth — but the manipulation has to match LCL's mechanism. Since the effect runs through the *set mean*, the intervention is on the surroundings' last feature.

**`make_slate_pair_lcl`** builds a slate from fixed $x_A$, $x_B$ and freshly-drawn fillers, with `others[:,-1] += context_shift`. Called twice per trial: once at `low=-1.5`, once at `high=+1.5`. The two slates share the same focal pair and differ in the ambient level of the last feature.

**`iia_ratio_recovery_lcl`** then computes:

- `d_mod = model_ab_gap(Xhi) - model_ab_gap(Xlo)` — the model's realised shift in the log-ratio
- `dC = Xhi.mean(axis=0) - Xlo.mean(axis=0)` — the *realised* change in set mean
- `d_true = (x_A - x_B) @ (A @ dC)` — the closed-form truth

The derivation of that truth: the gap is $x_A^\top(\theta + A\bar{x}) - x_B^\top(\theta + A\bar{x})$, so the $\theta$ terms are identical across the two slates and cancel, leaving $(x_A - x_B)^\top A\, \Delta\bar{x}$.

**The subtlety worth pausing on.** The fillers are *redrawn* between the low and high slates — they aren't the same vectors shifted. So the two slates differ in more than just the intended shift. This is fine, and in fact it's why `dC` is computed from the realised means rather than from the nominal $\pm 1.5$: under the true LCL model, the *only* channel by which fillers can affect the A–B gap is through $\bar{x}_S$, so conditioning on the realised mean makes the regression exactly correct rather than approximately correct. For a model that isn't LCL — attention, say — the redrawn fillers introduce extra variation the truth doesn't account for, which shows up as scatter around the fit. That inflates the standard error on the slope but does not bias it, since the extra variation is independent of `d_true`.

Note that `A` is rebuilt inside the probe via `make_theta_A(n_features)` rather than passed in. That's safe because `make_theta_A` is deterministic given `n_features` — but it's a coupling to be aware of. If the generator's `A` is ever customised at the call site, this probe would silently keep measuring against the default.


In [ ]:
def make_slate_pair_lcl(x_A, x_B, context_shift, rng, slate_size, n_features):
    others = rng.standard_normal((slate_size-2, n_features)).astype(np.float32)
    others[:,-1] += context_shift
    return np.vstack([x_A, x_B, others])

def model_ab_gap_lcl(model, X_np):
    feat = torch.tensor(X_np, dtype=torch.float32, device=device).unsqueeze(0)
    mask = torch.ones(1, X_np.shape[0], dtype=torch.bool, device=device)
    with torch.no_grad():
        lg = model(feat, mask)[0].cpu().numpy()
    return lg[0] - lg[1]

def iia_ratio_recovery_lcl(model, n_features, slate_size=5,
                           n_trials=3000, seed=7, low=-1.5, high=1.5):
    _, A = make_theta_A(n_features)
    rng  = np.random.default_rng(seed); model.eval()
    dt, dm = [], []
    for _ in range(n_trials):
        x_A = rng.standard_normal(n_features).astype(np.float32)
        x_B = rng.standard_normal(n_features).astype(np.float32)
        Xlo = make_slate_pair_lcl(x_A, x_B, low,  rng, slate_size, n_features)
        Xhi = make_slate_pair_lcl(x_A, x_B, high, rng, slate_size, n_features)
        d_mod  = model_ab_gap_lcl(model,Xhi) - model_ab_gap_lcl(model,Xlo)
        dC     = Xhi.mean(axis=0) - Xlo.mean(axis=0)
        d_true = (x_A-x_B) @ (A @ dC)
        dt.append(d_true); dm.append(d_mod)
    dt = np.array(dt); dm = np.array(dm)
    slope = np.polyfit(dt, dm, 1)[0]
    corr  = np.corrcoef(dt, dm)[0,1]
    return slope, corr

### Cell 16 — Smoke test: all three LCL models

The Part 2 counterpart to cell 7: one config (5000 sessions, 2 features), all three models, before committing to sweeps.

**What to check, again in priority order:**

1. **Is the yardstick's IIA near 1.0?** This validates the probe against the generator on the LCL side independently of Part 1. If it isn't ~1.0, suspect the set-mean convention first — that's where the two implementations could disagree.
2. **Does FATE beat attention?** The headline comparison for this half.
3. **Are the NLL gaps small?** With only one nonzero entry in $A$ and $a_{\text{strength}} = 0.3$, the planted effect is deliberately small. Expect *all three* NLL gaps to be tiny — which is exactly the setting where the identification failure is most visible, because predictive performance carries almost no information about whether the mechanism was learned.

**Expect attention to be unstable here.** Previous runs on LCL produced IIA values around $-0.2$ with a standard deviation near $0.5$ — meaning attention sometimes tracks the effect backwards, and its seed-to-seed variance exceeds the size of the effect it's meant to be recovering. If that reproduces, it isn't a bug; it's the finding. A single negative smoke-test number is not conclusive on its own, though — the sweeps with three seeds are what make it a claim.


In [ ]:
th2, A2 = make_theta_A(2)
df_lcl0 = generate_lcl_data(n_sessions=5000, n_features=2, theta=th2, A=A2, seed=0)
ds_lcl0 = SlateDataset(df_lcl0)

smoke_lcl = {}
for name, builder, epochs, lr in [
    ("Attention", lambda d: AttentionChoiceModelLCL(n_features=d, n_layers=1), 30, 1e-3),
    ("FATE",      lambda d: FATEChoiceModel(n_features=d, hidden=32),          40, 1e-3),
    ("Yardstick", lambda d: LCLYardstick(n_features=d),                       60, 5e-2),
]:
    m, nll = train_lcl_model(builder, ds_lcl0, epochs=epochs, lr=lr)
    iia, _ = iia_ratio_recovery_lcl(m, 2)
    smoke_lcl[name] = {"nll": nll, "iia": iia}
    print(f"{name:12s}  test_nll={nll:.4f}  iia_recovery={iia:.3f}")

nll_y = smoke_lcl["Yardstick"]["nll"]
print(f"\nNLL gap vs yardstick:")
for name in ["Attention","FATE"]:
    print(f"  {name}: {smoke_lcl[name]['nll']-nll_y:+.4f}")

### Cell 17 — LCL sweep engine (three models)

Structurally parallel to cell 8, with three differences worth naming.

**No `slope` key.** The old swap probe was a CDM-specific construction — it manipulates a neighbour and reads a raw logit, which doesn't map cleanly onto a set-average effect. So the LCL side carries only `iia` and `nll_gap`, and `lcl_sweep`'s accumulator is correspondingly narrower.

**Builders, not models.** Each entry in the config list is a lambda taking `d` and returning a fresh model, matching `train_lcl_model`'s interface. The lambdas close over `n_layers` from the enclosing scope, which is safe here because each is called immediately within the same iteration — but it's the kind of closure that becomes a late-binding bug if the list is ever built once outside the loop and reused.

**Only attention has depth.** `FATEChoiceModel` and `LCLYardstick` ignore `n_layers` entirely. So, exactly as with FETA in Part 1, FATE's curve in the depth panel is a flat control line whose wobble is pure seed noise — a free calibration for how much of any movement in attention's curve should be taken seriously.

Aggregation is the same as Part 1: mean and standard deviation across `SEEDS = [0, 1, 2]`, with a running printout per value.


In [ ]:
def run_lcl_config(n_sessions, n_features, n_layers, seed):
    th, A = make_theta_A(n_features)
    df    = generate_lcl_data(n_sessions=n_sessions, n_features=n_features,
                              theta=th, A=A, seed=seed)
    ds    = SlateDataset(df)
    out   = {}
    for name, builder, epochs, lr in [
        ("Attention", lambda d: AttentionChoiceModelLCL(n_features=d, n_layers=n_layers), 30, 1e-3),
        ("FATE",      lambda d: FATEChoiceModel(n_features=d, hidden=32),                 40, 1e-3),
        ("Yardstick", lambda d: LCLYardstick(n_features=d),                              60, 5e-2),
    ]:
        m, nll = train_lcl_model(builder, ds, epochs=epochs, lr=lr, seed=seed)
        iia, _ = iia_ratio_recovery_lcl(m, n_features, seed=200+seed)
        out[name] = {"nll": nll, "iia": iia}
    return out

def lcl_sweep(values, vary, base):
    accum = {m: {"iia":([],[]),"nll_gap":([],[])} for m in ["Attention","FATE"]}
    xs = []
    for v in values:
        cfg = dict(base); cfg[vary] = v
        per = {m: {"iia":[],"nll_gap":[]} for m in ["Attention","FATE"]}
        for sd in SEEDS:
            res   = run_lcl_config(seed=sd, **cfg)
            nll_y = res["Yardstick"]["nll"]
            for m in ["Attention","FATE"]:
                per[m]["iia"].append(res[m]["iia"])
                per[m]["nll_gap"].append(res[m]["nll"]-nll_y)
        xs.append(v)
        for m in ["Attention","FATE"]:
            for k in ["iia","nll_gap"]:
                accum[m][k][0].append(mean(per[m][k]))
                accum[m][k][1].append(stdev(per[m][k]) if len(per[m][k])>1 else 0.)
        print(f"{vary}={v} | Attn iia={accum['Attention']['iia'][0][-1]:.3f} "
              f"| FATE iia={accum['FATE']['iia'][0][-1]:.3f}")
    return xs, accum

### Cell 18 — Run the LCL sweeps

The same three axes as the CDM side — data size, feature count, attention depth — with LCL-appropriate base settings (`n_features=2`, `n_layers=1`).

Two things to watch in the printout:

- **Attention's IIA sign.** If values come back negative, attention is shifting the choice ratio in the *opposite* direction from the truth. That's a stronger failure than under-recovery and deserves flagging in the write-up as a distinct phenomenon, not just a smaller number.
- **FATE's stability across seeds.** The standard deviations are what make the comparison defensible. A high FATE mean with a wide spread is a much weaker result than a slightly lower mean with a tight one.

Expect this to take a similar amount of time to the CDM sweeps, or somewhat less — there's no `old_swap_slope` call on this side, and the LCL probe evaluates two slates per trial rather than looping one-at-a-time over items.


In [ ]:
print("=== LCL SWEEP 1: dataset size ===")
lxs_s, lres_s = lcl_sweep([10000,15000,20000], "n_sessions",
                           {"n_features":2,"n_layers":1})

print("\n=== LCL SWEEP 2: n_features ===")
lxs_f, lres_f = lcl_sweep([2,4,6], "n_features",
                           {"n_sessions":15000,"n_layers":1})

print("\n=== LCL SWEEP 3: n_layers (attention only) ===")
lxs_l, lres_l = lcl_sweep([1,2,3,4,6], "n_layers",
                           {"n_sessions":15000,"n_features":2})

### Cell 19 — LCL comparison plots

Same three-panel format as cell 10, with FATE in green (`C2`) against Attention in orange (`C1`), so the two figures can sit side by side in the thesis without a colour collision — FETA is blue, FATE is green, Attention is orange in both.

**The `ylim` caveat is far more likely to bite here.** The axis is fixed at `(-0.05, 1.15)`, and attention's LCL recovery has previously come in around $-0.2$. Those points will be **clipped off the bottom of the plot entirely**, along with the lower half of their error bars. If attention's orange line looks like it's hugging the floor or disappearing, change the limit to something like `(-0.6, 1.15)` and re-plot before drawing any conclusion. A clipped negative value read as "approximately zero" would understate the result substantially — negative recovery is a qualitatively different and more interesting failure than zero recovery.

As before, the dashed NLL-gap lines are on the twin axis and won't appear in the legend.

**What confirms the hypothesis on this side:** FATE's green line clearly above attention's orange one on the data-size and features panels, ideally with NLL gaps close together. The depth panel is the sharper test — if attention's recovery stays flat or falls as layers are added while FATE stays high, the capacity explanation is dead and the inductive-bias explanation is what remains.


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))

def plot_lcl_panel(ax, xs, accum, xlabel, title):
    for name,color in [("Attention","C1"),("FATE","C2")]:
        ax.errorbar(xs, accum[name]["iia"][0], yerr=accum[name]["iia"][1],
                    marker="o", label=f"{name} — IIA recovery", color=color)
    ax.axhline(1.0, ls=":", c="green", lw=1.2, label="perfect recovery (yardstick)")
    ax.set_xlabel(xlabel); ax.set_ylabel("IIA ratio recovery")
    ax.set_title(title); ax.set_ylim(-0.05,1.15)
    ax2 = ax.twinx()
    for name,color in [("Attention","C1"),("FATE","C2")]:
        ax2.errorbar(xs, accum[name]["nll_gap"][0], yerr=accum[name]["nll_gap"][1],
                     marker="^", ls="--", color=color, alpha=0.5,
                     label=f"{name} — NLL gap")
    ax2.set_ylabel("NLL gap vs yardstick (nats)", color="grey")
    ax2.tick_params(axis="y", labelcolor="grey")
    return ax2

plot_lcl_panel(axes[0], lxs_s, lres_s, "n_sessions", "LCL: recovery vs data size")
plot_lcl_panel(axes[1], lxs_f, lres_f, "n_features",  "LCL: recovery vs #features")
plot_lcl_panel(axes[2], lxs_l, lres_l, "n_layers",    "LCL: recovery vs attn depth")

handles,labels = axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",ncol=4,bbox_to_anchor=(0.5,-0.08))
plt.suptitle("LCL synthetic — FATE (matched inductive bias) vs Self-Attention", y=1.02)
plt.tight_layout()
plt.savefig("lcl_fate_vs_attn.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved lcl_fate_vs_attn.png")

### Cell 20 — Combined summary table

Flattens both halves into one long dataframe: `dataset` (CDM/LCL), `swept` (which axis), `value`, then paired columns for attention and for the matched model.

The `model_key` line is the small piece of glue that makes one loop cover both halves — the matched model is `FETA` for CDM and `FATE` for LCL, so the column is named `model_iia` generically with the actual name recorded in the `model` column. That keeps the table tidy in long format, which is what you want for plotting or for a `pivot` later.

The comparison to read straight off the table is `attn_iia` versus `model_iia` row by row. The number worth computing when writing this up is the *ratio* or *difference* between them, aggregated per sweep — that single number is the headline claim of the whole notebook.

`comparison_summary.csv` is written to the working directory. On Colab that's ephemeral, so download it or copy it to Drive before the runtime disconnects — along with `cdm_feta_vs_attn.png` and `lcl_fate_vs_attn.png` from cells 10 and 19.


In [ ]:
rows = []
for dataset, sweep_results in [
    ("CDM", [(xs_s,res_s,"n_sessions"),(xs_f,res_f,"n_features"),(xs_l,res_l,"n_layers")]),
    ("LCL", [(lxs_s,lres_s,"n_sessions"),(lxs_f,lres_f,"n_features"),(lxs_l,lres_l,"n_layers")]),
]:
    model_key = "FETA" if dataset=="CDM" else "FATE"
    for xs, accum, swept in sweep_results:
        for i,v in enumerate(xs):
            rows.append({
                "dataset": dataset, "swept": swept, "value": v,
                "attn_iia":  round(accum["Attention"]["iia"][0][i],4),
                "model_iia": round(accum[model_key]["iia"][0][i],4),
                "model":     model_key,
                "attn_nll_gap":  round(accum["Attention"]["nll_gap"][0][i],4),
                "model_nll_gap": round(accum[model_key]["nll_gap"][0][i],4),
            })

summary = pd.DataFrame(rows)
summary.to_csv("comparison_summary.csv", index=False)
print(summary.to_string(index=False))
print("\nsaved comparison_summary.csv")

### Cell 21 — Reading the results

#### If the hypothesis holds

**If FETA consistently beats Attention on CDM IIA recovery, and FATE consistently beats Attention on LCL IIA recovery**, this is the inductive-bias hypothesis confirmed under controlled conditions. The argument it supports:

- Self-attention *can* represent these context effects — it has ample capacity, and the depth sweep shows adding more doesn't help. But the cross-entropy objective cannot teach it *which* effect to represent. That's the identification failure established in the previous notebook.
- A model whose architecture matches the true effect structure recovers that effect from the **same data** under the **same objective**, because the functional form constrains the solution space in the right direction.
- So the question was never "does the model need to be bigger." It was "does the model need the right inductive bias."

The fact that this holds across two *different* effect structures, each with its own matched architecture, is what makes it a claim about matching rather than a claim about FETA or FATE specifically.

#### If it doesn't

Two other outcomes are informative and shouldn't be treated as failed runs:

- **Matched models recover no better than attention.** Then the bottleneck isn't architectural at all — which points back at the training signal, consistent with the auxiliary-loss experiment where an explicit IIA term recovered nearly the full effect. Architecture and objective would then be two routes to the same fix, with the objective being the more general one.
- **Matched models recover better but still fall short of 1.0.** The most likely outcome, and arguably the most interesting: it says inductive bias helps *and* something else is still binding. The gap between the matched model and the yardstick would then be the quantity worth explaining, and it isolates "right conditioning structure but the interaction form still has to be learned" as the residual difficulty.

#### Where this leads

Both confirming and partial outcomes motivate the same next step. Imposing the right bias through architecture requires **knowing the effect structure in advance** — and on real data like Expedia, nobody does. FETA works on CDM because we planted a pairwise effect; FATE works on LCL because we planted a set-average one. Neither is a usable recipe when the truth is unknown.

That is what the target-conditioned formulation is for:

$$A_{c,S} = g\big(\text{embed}(c), \{\text{embed}(j)\}_{j \in S}\big)$$

Instead of hard-coding the interaction structure, the interaction is *learned* by conditioning it on the target and the candidate set. The experiments in this notebook establish the case for why that's necessary: they show that structure matters and that a general-purpose architecture won't find it on its own. The open contribution is learning the structure rather than assuming it.
